# 03. Object Storage – MinIO

## Mục đích

Quy trình này kiểm tra và quản lý dữ liệu đã được lưu trữ trên
Object Storage MinIO sau bước Data Ingestion.

Các nội dung thực hiện:

1. Kiểm tra kết nối MinIO.
2. Kiểm tra bucket `ecommerce-lakehouse`.
3. Kiểm tra cấu trúc Object Storage.
4. Kiểm tra các CSV trong `raw/`.
5. Kiểm tra ảnh sản phẩm trong `images/products/`.
6. Kiểm tra ingestion log.
7. Kiểm tra metadata và dung lượng object.
8. Đọc thử dữ liệu CSV từ MinIO.
9. Đọc thử ảnh từ MinIO.
10. Kiểm tra tính đầy đủ của dữ liệu sau ingestion.
11. Xác nhận Object Storage sẵn sàng cho bước xử lý tiếp theo.

Lưu ý:
Dữ liệu không được upload lại trong quy trình này.
Phần upload đã được thực hiện ở quy trình Data Ingestion.

In [1]:
import os
from pathlib import Path
from io import BytesIO

import pandas as pd
from dotenv import load_dotenv
from minio import Minio

In [2]:
BASE_DIR = Path.cwd().resolve()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent


ENV_FILE = BASE_DIR / ".env"

if not ENV_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy .env: "
        f"{ENV_FILE}"
    )

load_dotenv(
    ENV_FILE
)

print(
    "Project root:",
    BASE_DIR
)

Project root: D:\E-commerce-Data-Lakehouse


In [3]:
MINIO_ENDPOINT = os.getenv(
    "MINIO_ENDPOINT",
    "localhost:9000"
)

MINIO_ACCESS_KEY = os.getenv(
    "MINIO_ROOT_USER"
)

MINIO_SECRET_KEY = os.getenv(
    "MINIO_ROOT_PASSWORD"
)

if not MINIO_ACCESS_KEY or not MINIO_SECRET_KEY:
    raise EnvironmentError(
        "Thiếu MINIO_ROOT_USER hoặc "
        "MINIO_ROOT_PASSWORD trong .env"
    )


client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print(
    "✓ MinIO client initialized"
)

✓ MinIO client initialized


In [4]:
try:

    buckets = list(
        client.list_buckets()
    )

    print(
        "✓ Kết nối MinIO thành công"
    )

    print(
        f"Số bucket hiện có: "
        f"{len(buckets)}"
    )

except Exception as e:

    raise ConnectionError(
        f"Không thể kết nối MinIO tại "
        f"{MINIO_ENDPOINT}: {e}"
    )

✓ Kết nối MinIO thành công
Số bucket hiện có: 1


In [5]:
BUCKET_NAME = "ecommerce-lakehouse"

if not client.bucket_exists(
    BUCKET_NAME
):

    raise RuntimeError(
        f"Không tìm thấy bucket: "
        f"{BUCKET_NAME}"
    )


print(
    f"✓ Bucket tồn tại: "
    f"{BUCKET_NAME}"
)

✓ Bucket tồn tại: ecommerce-lakehouse


In [6]:
objects = list(
    client.list_objects(
        BUCKET_NAME,
        recursive=True
    )
)

print(
    f"Tổng số object trong bucket: "
    f"{len(objects):,}"
)


for obj in objects[:20]:

    print(
        "-",
        obj.object_name
    )


if len(objects) > 20:

    print(
        f"... và "
        f"{len(objects) - 20:,} "
        f"object khác"
    )

Tổng số object trong bucket: 12
- images/products/product_001.jpg
- images/products/product_002.jpg
- images/products/product_003.jpg
- images/products/product_004.jpg
- images/products/product_005.jpg
- landing/customers/customers.csv
- landing/order_items/order_items.csv
- landing/orders/orders.csv
- landing/payments/payments.csv
- landing/products/products.csv
- landing/reviews/reviews.csv
- logs/ingestion/ingestion_log_20260930T153809Z.json


In [7]:
landing_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="landing/",
        recursive=True
    )
)

image_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="images/products/",
        recursive=True
    )
)

log_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="logs/ingestion/",
        recursive=True
    )
)

warehouse_objects = list(
    client.list_objects(
        BUCKET_NAME,
        prefix="warehouse/",
        recursive=True
    )
)


print(
    f"Landing CSV objects : "
    f"{len(landing_objects):,}"
)

print(
    f"Product images      : "
    f"{len(image_objects):,}"
)

print(
    f"Ingestion logs      : "
    f"{len(log_objects):,}"
)

print(
    f"Warehouse objects   : "
    f"{len(warehouse_objects):,}"
)

Landing CSV objects : 6
Product images      : 5
Ingestion logs      : 1
Warehouse objects   : 0


In [9]:
EXPECTED_LANDING_OBJECTS = {
    "landing/customers/customers.csv",

    "landing/products/products.csv",

    "landing/orders/orders.csv",

    "landing/order_items/order_items.csv",

    "landing/reviews/reviews.csv",

    "landing/payments/payments.csv"
}


actual_landing_objects = {
    obj.object_name
    for obj in landing_objects
}


missing_landing_objects = (
    EXPECTED_LANDING_OBJECTS
    - actual_landing_objects
)


unexpected_landing_objects = (
    actual_landing_objects
    - EXPECTED_LANDING_OBJECTS
)


print(
    "Expected Landing CSV:",
    len(EXPECTED_LANDING_OBJECTS)
)

print(
    "Actual Landing CSV:",
    len(actual_landing_objects)
)

print(
    "Missing:",
    missing_landing_objects
)

print(
    "Unexpected:",
    unexpected_landing_objects
)

Expected Landing CSV: 6
Actual Landing CSV: 6
Missing: set()
Unexpected: set()


In [10]:
print(
    f"Tổng số ảnh trên MinIO: "
    f"{len(image_objects):,}"
)


if len(image_objects) == 0:

    print(
        "✗ Không tìm thấy "
        "ảnh sản phẩm"
    )

else:

    print(
        "✓ Có dữ liệu "
        "ảnh sản phẩm"
    )

    print(
        "\nMột số object mẫu:"
    )

    for obj in image_objects[:10]:

        print(
            "-",
            obj.object_name,
            "|",
            obj.size,
            "bytes"
        )

Tổng số ảnh trên MinIO: 5
✓ Có dữ liệu ảnh sản phẩm

Một số object mẫu:
- images/products/product_001.jpg | 181972 bytes
- images/products/product_002.jpg | 229635 bytes
- images/products/product_003.jpg | 41352 bytes
- images/products/product_004.jpg | 36381 bytes
- images/products/product_005.jpg | 126543 bytes


In [11]:
object_metadata = []

for obj in objects:

    if obj.object_name.startswith(
        "landing/"
    ):

        data_type = (
            "LANDING_CSV"
        )

    elif obj.object_name.startswith(
        "images/products/"
    ):

        data_type = (
            "IMAGE"
        )

    elif obj.object_name.startswith(
        "warehouse/"
    ):

        data_type = (
            "LAKEHOUSE"
        )

    elif obj.object_name.startswith(
        "logs/"
    ):

        data_type = (
            "LOG"
        )

    else:

        data_type = (
            "OTHER"
        )


    object_metadata.append({
        "object_name":
            obj.object_name,

        "data_type":
            data_type,

        "size_bytes":
            obj.size,

        "size_mb":
            round(
                obj.size
                / (1024 * 1024),
                4
            ),

        "last_modified":
            obj.last_modified
    })


metadata_df = pd.DataFrame(
    object_metadata
)

metadata_df.head(20)

,object_name,data_type,size_bytes,size_mb,last_modified
0,images/products/product_001.jpg,IMAGE,181972,0.1735,2026-09-30 15:38:11.690000+00:00
1,images/products/product_002.jpg,IMAGE,229635,0.2190,2026-09-30 15:38:11.741000+00:00
2,images/products/product_003.jpg,IMAGE,41352,0.0394,2026-09-30 15:38:11.789000+00:00
3,images/products/product_004.jpg,IMAGE,36381,0.0347,2026-09-30 15:38:11.837000+00:00
4,images/products/product_005.jpg,IMAGE,126543,0.1207,2026-09-30 15:38:11.843000+00:00
5,landing/customers/customers.csv,LANDING_CSV,8661712,8.2605,2026-09-30 15:38:11.096000+00:00
6,landing/order_items/order_items.csv,LANDING_CSV,15024227,14.3282,2026-09-30 15:38:11.425000+00:00
7,landing/orders/orders.csv,LANDING_CSV,17506326,16.6953,2026-09-30 15:38:11.302000+00:00
8,landing/payments/payments.csv,LANDING_CSV,5737249,5.4715,2026-09-30 15:38:11.666000+00:00
9,landing/products/products.csv,LANDING_CSV,2828882,2.6978,2026-09-30 15:38:11.138000+00:00


In [12]:
if metadata_df.empty:

    storage_summary = (
        pd.DataFrame(
            columns=[
                "data_type",
                "object_count",
                "total_size_bytes",
                "total_size_mb"
            ]
        )
    )

else:

    storage_summary = (
        metadata_df
        .groupby(
            "data_type"
        )
        .agg(
            object_count=(
                "object_name",
                "count"
            ),

            total_size_bytes=(
                "size_bytes",
                "sum"
            )
        )
        .reset_index()
    )


    storage_summary[
        "total_size_mb"
    ] = (
        storage_summary[
            "total_size_bytes"
        ]
        / (1024 * 1024)
    ).round(2)


storage_summary

,data_type,object_count,total_size_bytes,total_size_mb
0,IMAGE,5,615883,0.59
1,LANDING_CSV,6,63813156,60.86
2,LOG,1,4475,0.00


In [13]:
ORDERS_OBJECT = (
    "landing/orders/orders.csv"
)


response = client.get_object(
    BUCKET_NAME,
    ORDERS_OBJECT
)


try:

    data = (
        response.read()
    )

finally:

    response.close()

    response.release_conn()


orders = pd.read_csv(
    BytesIO(data)
)


print(
    "✓ Đọc orders.csv "
    "thành công"
)

print(
    "Số dòng:",
    len(orders)
)

print(
    "Số cột:",
    len(orders.columns)
)


orders.head()

✓ Đọc orders.csv thành công
Số dòng: 99441
Số cột: 8


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [15]:
orders_check = {
    "object":
        ORDERS_OBJECT,

    "rows":
        len(orders),

    "columns":
        len(
            orders.columns
        ),

    "empty":
        orders.empty,

    "duplicate_rows":
        int(
            orders
            .duplicated()
            .sum()
        ),

    "order_id_null":
        int(
            orders[
                "order_id"
            ]
            .isna()
            .sum()
        ),

    "order_id_duplicate":
        int(
            orders[
                "order_id"
            ]
            .duplicated()
            .sum()
        )
}


pd.DataFrame(
    [orders_check]
)

,object,rows,columns,empty,duplicate_rows,order_id_null,order_id_duplicate
0,landing/orders/orders.csv,99441,8,False,0,0,0


In [16]:
if not image_objects:

    raise RuntimeError(
        "Không có ảnh để kiểm tra"
    )


sample_image_object = (
    image_objects[0]
    .object_name
)


print(
    "Ảnh kiểm tra:",
    sample_image_object
)

Ảnh kiểm tra: images/products/product_001.jpg


In [17]:
response = client.get_object(
    BUCKET_NAME,
    sample_image_object
)


try:

    image_data = (
        response.read()
    )

finally:

    response.close()

    response.release_conn()


print(
    "✓ Đọc ảnh thành công"
)

print(
    "Dung lượng:",
    len(image_data),
    "bytes"
)

✓ Đọc ảnh thành công
Dung lượng: 181972 bytes


In [18]:
empty_image_objects = [
    obj
    for obj in image_objects
    if obj.size == 0
]


print(
    "Ảnh 0 byte:",
    len(
        empty_image_objects
    )
)


if not empty_image_objects:

    print(
        "✓ Không phát hiện "
        "object ảnh rỗng"
    )

else:

    for obj in empty_image_objects[:20]:

        print(
            "-",
            obj.object_name
        )

Ảnh 0 byte: 0
✓ Không phát hiện object ảnh rỗng


In [19]:
print(
    f"Số ingestion log: "
    f"{len(log_objects)}"
)


if not log_objects:

    print(
        "✗ Không tìm thấy "
        "ingestion log"
    )

else:

    print(
        "✓ Tìm thấy "
        "ingestion log"
    )

    print(
        "\n5 log gần nhất:"
    )

    for obj in log_objects[-5:]:

        print(
            "-",
            obj.object_name
        )

Số ingestion log: 1
✓ Tìm thấy ingestion log

5 log gần nhất:
- logs/ingestion/ingestion_log_20260930T153809Z.json


In [20]:
prefix_expectations = {
    "landing/":
        True,

    "images/":
        True,

    "logs/":
        True,

    # Warehouse chưa bắt buộc
    # có dữ liệu ở bước hiện tại
    "warehouse/":
        False
}


prefix_status = []


for (
    prefix,
    expected_now
) in prefix_expectations.items():

    prefix_objects = list(
        client.list_objects(
            BUCKET_NAME,
            prefix=prefix,
            recursive=True
        )
    )

    object_count = len(
        prefix_objects
    )


    if object_count > 0:

        status = (
            "AVAILABLE"
        )

    elif expected_now:

        status = (
            "MISSING"
        )

    else:

        status = (
            "NOT_YET"
        )


    prefix_status.append({
        "prefix":
            prefix,

        "object_count":
            object_count,

        "expected_now":
            expected_now,

        "status":
            status
    })


prefix_status_df = pd.DataFrame(
    prefix_status
)

prefix_status_df

,prefix,object_count,expected_now,status
0,landing/,6,True,AVAILABLE
1,images/,5,True,AVAILABLE
2,logs/,1,True,AVAILABLE
3,warehouse/,0,False,NOT_YET


In [21]:
landing_ready = (
    len(
        missing_landing_objects
    ) == 0

    and

    len(
        actual_landing_objects
    )
    ==
    len(
        EXPECTED_LANDING_OBJECTS
    )
)


images_ready = (
    len(
        image_objects
    ) > 0

    and

    len(
        empty_image_objects
    ) == 0
)


logs_ready = (
    len(
        log_objects
    ) > 0
)


storage_status = (
    "READY"

    if (
        landing_ready
        and images_ready
        and logs_ready
    )

    else "NOT_READY"
)


print(
    "========== OBJECT STORAGE SUMMARY =========="
)


print(
    "Bucket:",
    BUCKET_NAME
)


print(
    "\n--- LANDING CSV ---"
)

print(
    "Expected:",
    len(
        EXPECTED_LANDING_OBJECTS
    )
)

print(
    "Actual  :",
    len(
        actual_landing_objects
    )
)

print(
    "Missing :",
    len(
        missing_landing_objects
    )
)


print(
    "\n--- IMAGES ---"
)

print(
    "Objects :",
    len(
        image_objects
    )
)

print(
    "Empty   :",
    len(
        empty_image_objects
    )
)


print(
    "\n--- LOGS ---"
)

print(
    "Ingestion logs:",
    len(
        log_objects
    )
)


print(
    "\n--- WAREHOUSE ---"
)

print(
    "Objects:",
    len(
        warehouse_objects
    )
)

print(
    "Required now:",
    "NO"
)


print(
    "\nStorage status:",
    storage_status
)

========== OBJECT STORAGE SUMMARY ==========
Bucket: ecommerce-lakehouse

--- LANDING CSV ---
Expected: 6
Actual  : 6
Missing : 0

--- IMAGES ---
Objects : 5
Empty   : 0

--- LOGS ---
Ingestion logs: 1

--- WAREHOUSE ---
Objects: 0
Required now: NO

Storage status: READY
